# TARDIS_MODEL

---

> **Auteur**: Laurince AGANI    
> **Dataset**: `cleaned_dataset.csv`    
> **Objectif**: Prédire le retard moyen à l'arrivée de tous les trains (`Average delay of all trains at arrival`) en minutes, et comprendre les causes des dysfonctionnements opérationnels.

---

### Plan du notebook    
1. Importation des librairies nécessaires et configurations 
2. Chargement et inspection des données

## 1. Importation des librairies nécessaires et configurations


In [7]:
# Manipulation de données
import numpy as np
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn: Préprocessing
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Utilitaires
import warnings
warnings.filterwarnings("ignore")

## 2. Chargement et inspection des données

In [4]:
# Chargement du dataset
df = pd.read_csv(
    "cleaned_dataset.csv",
    sep=';',
    encoding="utf-8"
)

# Inspection initiale du dataset
print('=' * 84)
print("APERÇU DES 5 PREMIÈRES LIGNES")
print('=' * 84)
display(df.head(5))

# Forme du dataser
print("Dimensions du dataset")
print(f"    {df.shape[0]} observations")
print(f"    {df.shape[1]} features")

APERÇU DES 5 PREMIÈRES LIGNES


,Date,Service,Departure station,Arrival station,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,Number of trains delayed at arrival,Average delay of late trains at arrival,Average delay of all trains at arrival,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",Year,Month,Delay categories,is_delayed,Cancellation_rate
0,2018-01,National,BORDEAUX ST JEAN,PARIS MONTPARNASSE,141.00,870.0,5.0,289.0,11.247809,3.693179,147.0,28.436735,6.511118,110.0,6.510000,44.0,8.0,36.134454,31.092437,10.924370,15.966387,5.040000,0.840336,2018,1,average delay,1,0.005747
1,2018-01,National,LE MANS,PARIS MONTPARNASSE,56.00,406.0,1.0,213.0,8.479969,4.567119,105.0,18.049000,5.363539,32.0,5.363539,9.0,4.0,20.000000,35.000000,16.666667,16.666667,8.333333,3.333333,2018,1,average delay,1,0.002463
2,2018-01,National,PARIS MONTPARNASSE,LA ROCHELLE VILLE,166.00,226.0,0.0,21.0,6.239683,0.286283,19.0,24.736842,2.938053,11.0,2.938053,6.0,1.0,22.222222,27.777778,16.666667,16.666667,5.555556,11.111111,2018,1,minor delay,0,0.000000
3,2018-01,National,PARIS MONTPARNASSE,NANTES,216.21,508.0,3.0,71.0,7.235211,0.980000,58.0,33.726437,5.292211,39.0,5.292211,18.0,3.0,33.333333,22.222222,16.666667,20.370370,5.555556,1.851852,2018,1,average delay,1,0.005906
4,2018-01,National,POITIERS,PARIS MONTPARNASSE,94.00,472.0,4.0,224.0,6.784673,3.229701,89.0,14.592697,4.882372,42.0,4.882372,10.0,0.0,15.789474,45.614035,18.750000,15.789474,1.754386,1.754386,2018,1,minor delay,0,0.008475


Dimensions du dataset
    11487 observations
    28 features


In [ ]:
# Informations sur le dataset
print('=' * 84)
print("INFORMATIONS SUR LE DATASET")
print('=' * 84)
df.info()

INFORMATIONS SUR LE DATASET
<class 'pandas.DataFrame'>
RangeIndex: 11487 entries, 0 to 11486
Data columns (total 28 columns):
 #   Column                                                                         Non-Null Count  Dtype  
---  ------                                                                         --------------  -----  
 0   Date                                                                           11487 non-null  str    
 1   Service                                                                        11487 non-null  str    
 2   Departure station                                                              11487 non-null  str    
 3   Arrival station                                                                11487 non-null  str    
 4   Average journey time                                                           11487 non-null  float64
 5   Number of scheduled trains                                                     11487 non-null  float64
 6   Numbe

In [9]:
# Statistiques descriptives du dataset
print('=' * 84)
print("STATISTIQUES DESCRIPTIVES DU DATASET")
print('=' * 84)
df.describe().round(2)

STATISTIQUES DESCRIPTIVES DU DATASET


,Average journey time,Number of scheduled trains,Number of cancelled trains,Number of trains delayed at departure,Average delay of late trains at departure,Average delay of all trains at departure,Number of trains delayed at arrival,Average delay of late trains at arrival,Average delay of all trains at arrival,Number of trains delayed > 15min,Average delay of trains > 15min (if competing with flights),Number of trains delayed > 30min,Number of trains delayed > 60min,Pct delay due to external causes,Pct delay due to infrastructure,Pct delay due to traffic management,Pct delay due to rolling stock,Pct delay due to station management and equipment reuse,"Pct delay due to passenger handling (crowding, disabled persons, connections)",Year,Month,is_delayed,Cancellation_rate
count,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11487.00,11477.00
mean,170.72,269.85,8.52,86.13,12.28,3.11,37.23,35.17,6.03,26.72,36.24,12.41,5.11,21.52,21.83,20.34,18.82,7.33,7.52,2021.48,6.51,0.55,inf
std,87.16,181.15,22.49,89.89,11.80,5.15,30.98,15.60,7.04,22.43,19.56,11.97,5.89,15.81,14.74,14.52,13.35,7.93,9.36,2.31,3.45,0.50,NaN
min,0.00,0.00,0.00,0.00,0.00,-229.27,0.00,-40.11,-472.64,0.00,-2.71,-44.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,2018.00,1.00,0.00,0.00
25%,101.00,152.00,0.00,22.00,6.28,1.22,15.00,25.95,3.43,11.00,28.15,4.00,1.00,10.71,12.16,10.53,10.19,0.00,0.00,2019.00,4.00,0.00,0.00
50%,164.00,229.00,2.00,52.00,10.38,2.33,29.00,33.47,5.35,21.00,37.54,9.00,3.00,19.05,20.00,18.75,17.11,5.88,5.00,2021.00,7.00,1.00,0.01
75%,222.00,351.00,7.00,125.00,15.57,3.89,50.00,42.25,8.00,36.00,46.63,17.00,7.00,29.41,29.41,28.18,25.00,10.87,11.11,2024.00,9.00,1.00,0.03
max,786.00,1100.00,297.00,1066.00,316.19,115.05,376.00,299.60,92.00,312.00,299.60,202.00,71.00,100.00,100.00,100.00,100.00,100.00,100.00,2025.00,12.00,1.00,inf


In [13]:
# Analyse des valeurs manquantes

# Afficher les colonnes avec des valeurs manquantes
missing = df.isnull().sum()
missing_pct = ((df.isnull().sum() * 100) / len(df))

missing_df = pd.DataFrame({
    "Valeurs manquantes": missing,
    "Pourcentage (%)": missing_pct.round(2)
}).sort_values("Pourcentage (%)", ascending=False)

missing_with_nan = missing_df[missing_df["Valeurs manquantes"] > 0]

print('=' * 84)
print("VALEURS MANQUANTES PAR COLONNE")
print('=' * 84)
if len(missing_with_nan) == 0:
    print("Aucune valeur manquante détectée !")
else:
    display(missing_with_nan)
    print(f"{len(missing_with_nan)} colonnes contiennent des valeurs manquantes.")

VALEURS MANQUANTES PAR COLONNE


,Valeurs manquantes,Pourcentage (%)
Cancellation_rate,10,0.09


1 colonnes contiennent des valeurs manquantes.
